# 06 Outcomes and impact

Maps every method to the same decision policy, builds the applied series, and computes the locked energy terms, the slot-level agreement and the site-day minimum-demand impact into one site-day decision-and-impact table.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** The interval tables of notebooks 03, 04 and 05, the M9 site-day table, and the datasets (for the recorded net load and labels).

**Outputs.** `outputs/01_final_evaluation/06_site_days/site_days.parquet` (one row per method × site-day) and `site_day_decisions.csv` (the decisions without the energy detail); `manifests/06_outcomes.json`.

**Approximate runtime.** About three minutes.

**Prerequisites.** Notebooks 01, 03, 05 and, for a release, 04.

**Main process.**

1. Join each interval table with the recorded net load and labels.
2. Outcomes: M7 and M8 are AUTO_CORRECT when any slot is flagged, AUTO_KEEP otherwise; M9 is three-way from its probability.
3. Energy per slot is 2·y·0.25 MWh; proposed, required and correctly corrected energies; slot confusion counts; window IoU; boundary errors; raw and applied daily minimum demand for the method and for the reference correction.
4. Check that all methods share identical (cohort, station, date) keys.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Build the site-day table

If a method's predictions are missing the stage continues with the others and says so; such a run is a debugging run, not a release.

In [ ]:
site_days = cli.stage_outcomes(SETTINGS)
display(site_days.head())

## 3. Outcome rates and minimum-demand impact

Outcome shares per method and cohort on headline-confidence days, and the number of corrected days on which the daily minimum changed, with the mean signed change in MW.

In [ ]:
t = site_days[site_days["headline"]]
rates = pd.crosstab([t["method"], t["cohort"]], t["outcome"], normalize="index").round(3)
display(rates)
applied = t[t["outcome"] == "AUTO_CORRECT"]
impact = (applied.assign(changed=applied["min_change_mw"].abs() > 0)
          .groupby(["method", "cohort"]).agg(corrected_days=("date", "size"), min_changed_days=("changed", "sum"),
                                              mean_min_change_mw=("min_change_mw", "mean")).round(3))
display(impact)

## Conclusion

The site-day table is the single input of the metrics in notebook 07 and of every figure in notebook 08.